# Password-attack signs (failed attempts, unusual times, unknown devices)

# Coding problem, 1.2.a: Detect brute-force login attempts

**Skill:** Detect Attacks · **Maps to:** 1.2.A · **Language:** Python 3 (pytest)

## Problem
Build the detector a login-monitoring system would use. Given a list of authentication events,
return every username that has five or more failed login attempts within any 60-second window.

This is the core algorithm behind real intrusion-detection rules (e.g., fail2ban's default
policy). The defender's goal is to catch automated password-guessing tools before they succeed.

## Function contract
Implement `solve(events: list[dict]) -> list[str]` in the code editor (see the Reference section below). Return the sorted, de-duplicated list of usernames flagged as brute-force
targets.

## Input
A list of event dicts. Each dict has:
- `ts` (`int`), Unix timestamp in seconds of the login attempt
- `user` (`str`), the username targeted
- `ok` (`bool`), `True` if the login succeeded, `False` if it failed

Events may arrive in any order (do not assume sorted by timestamp).

## Output
A sorted `list[str]` of unique usernames that triggered the brute-force threshold: 5 or more
**failed** logins (where `ok == False`) within any 60-second window. A window is valid when
`max_ts - min_ts < 60` (strict less-than, so exactly 60 is NOT within the window). Successes
are ignored for counting purposes.

## Examples
```
events = [
    {"ts": 1000, "user": "alice", "ok": False},
    {"ts": 1010, "user": "alice", "ok": False},
    {"ts": 1020, "user": "alice", "ok": False},
    {"ts": 1030, "user": "alice", "ok": False},
    {"ts": 1040, "user": "alice", "ok": False},  # 5 failures in 40s -> flagged
    {"ts": 1000, "user": "bob",   "ok": False},
    {"ts": 1100, "user": "bob",   "ok": False},  # 2 failures, not enough -> not flagged
]
solve(events) -> ["alice"]
```

```
solve([]) -> []
```

```
# Four failures in window: below threshold
events = [{"ts": i*10, "user": "carol", "ok": False} for i in range(4)]
solve(events) -> []
```

## Constraints
- Deterministic, offline, standard library only.
- Events may be unsorted; your solution must handle any ordering.
- The 60-second window: at least 5 failures where `max_ts - min_ts < 60`.
- Must pass all cases in the tests (visible + hidden).

## Reference
Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

### Entry point
```python
def solve(events: list[dict]) -> list[str]
```

### Input (`events`)
A `list[dict]`. Each dict is one authentication event with exactly these keys:
- `ts`   (`int`), Unix timestamp in whole seconds
- `user` (`str`), username targeted by the attempt
- `ok`   (`bool`), `True` = login succeeded, `False` = login failed

The list may be empty. Events may appear in any order (unsorted by timestamp).

### Output (`result`)
A **sorted, de-duplicated** `list[str]` of usernames that are flagged as brute-force targets.

### Detection rule (precise)
A username is flagged if and only if there exist at least 5 events for that user where:
1. `ok == False` (failed logins only; successes do NOT count toward the threshold)
2. `max(ts) - min(ts) < 60` across those 5 (or more) events (strict less-than; the window is
   open on the right, spans of exactly 60 seconds are NOT within the window)

Algorithm: for each username, collect all timestamps where `ok == False`, sort them, then use a
sliding window: for each position `i`, check if `ts[i+4] - ts[i] < 60`; if so, flag the user.

### Guarantees / rules
- Pure function, deterministic, no I/O, no network. Standard library only.
- Output always sorted ascending and de-duplicated.
- Empty input returns `[]`.

### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).


In [ ]:
# Your implementation
"""Starter code for 1.2.a. Student completes the TODO.
This file intentionally FAILS the tests until implemented."""
from typing import List, Dict, Any


def solve(events: List[Dict[str, Any]]) -> List[str]:
    """Return sorted, de-duplicated usernames with >=5 failed logins in any 60s window.

    Args:
        events: list of dicts, each with keys:
                  ts   (int)  - Unix timestamp in seconds
                  user (str)  - username targeted
                  ok   (bool) - True=success, False=failure
    Returns:
        Sorted list of usernames flagged as brute-force targets.
        See the problem statement above and the Reference section below for the exact detection rule.
    """
    # TODO: for each user, collect failure timestamps, sort them, then use a
    #       sliding window of size 5 to check if ts[i+4] - ts[i] < 60.
    #       Return a sorted list of flagged usernames.
    raise NotImplementedError("implement solve() — see the problem statement above")


## Self-check tests
Run the cell below to check your `solve` implementation above.

In [ ]:
import __main__ as impl  # the functions defined above

# ---- helpers -------------------------------------------------------
def _e(ts, user, ok):
    return {"ts": ts, "user": user, "ok": ok}


# ---- visible cases -------------------------------------------------------
def test_single_user_flagged():
    events = [
        _e(1000, "alice", False),
        _e(1010, "alice", False),
        _e(1020, "alice", False),
        _e(1030, "alice", False),
        _e(1040, "alice", False),  # 5 failures in 40s -> flagged
    ]
    assert impl.solve(events) == ["alice"]


def test_two_users_one_flagged():
    events = [
        _e(1000, "alice", False),
        _e(1010, "alice", False),
        _e(1020, "alice", False),
        _e(1030, "alice", False),
        _e(1040, "alice", False),
        _e(1000, "bob",   False),
        _e(1100, "bob",   False),  # only 2 failures, spread across >60s -> not flagged
    ]
    assert impl.solve(events) == ["alice"]


def test_empty():
    assert impl.solve([]) == []


def test_all_successes_not_flagged():
    events = [_e(1000 + i * 5, "carol", True) for i in range(10)]
    assert impl.solve(events) == []


# ---- hidden / edge cases -------------------------------------------------
def test_exactly_four_failures_not_flagged():
    events = [_e(1000 + i * 10, "dave", False) for i in range(4)]
    assert impl.solve(events) == []


def test_window_boundary_exactly_60_not_flagged():
    # 5 events spanning exactly 60 seconds: ts[0]=0, ts[4]=60 -> NOT flagged (strict <60)
    events = [_e(i * 15, "eve", False) for i in range(5)]  # 0,15,30,45,60 -> span=60 -> NOT flagged
    assert impl.solve(events) == []


def test_window_boundary_just_inside_flagged():
    # 5 events spanning 59 seconds -> flagged
    events = [
        _e(0,  "frank", False),
        _e(15, "frank", False),
        _e(30, "frank", False),
        _e(45, "frank", False),
        _e(59, "frank", False),  # span = 59 < 60 -> flagged
    ]
    assert impl.solve(events) == ["frank"]


def test_unsorted_events_handled():
    # Events arrive out of order; solution must sort
    events = [
        _e(1040, "grace", False),
        _e(1000, "grace", False),
        _e(1020, "grace", False),
        _e(1030, "grace", False),
        _e(1010, "grace", False),
    ]
    assert impl.solve(events) == ["grace"]


def test_success_in_middle_not_counted():
    # 4 failures + 1 success in window -> only 4 failures counted -> not flagged
    events = [
        _e(1000, "henry", False),
        _e(1010, "henry", False),
        _e(1020, "henry", True),   # success -- not counted
        _e(1030, "henry", False),
        _e(1040, "henry", False),
    ]
    assert impl.solve(events) == []


def test_multiple_users_both_flagged_sorted():
    events = (
        [_e(2000 + i * 5, "zara", False) for i in range(5)] +
        [_e(3000 + i * 5, "alice", False) for i in range(5)]
    )
    result = impl.solve(events)
    assert result == ["alice", "zara"]
    assert result == sorted(result)


def test_second_window_triggers_flag():
    # First 5 events spread beyond 60s, but a LATER group of 5 within 60s
    events = [
        _e(0,   "iris", False),
        _e(70,  "iris", False),
        _e(140, "iris", False),
        _e(210, "iris", False),
        _e(280, "iris", False),  # window 0-280 -> 280s -> not flagged
        # Second cluster within 60s
        _e(400, "iris", False),
        _e(410, "iris", False),
        _e(420, "iris", False),
        _e(430, "iris", False),
        _e(440, "iris", False),  # window 400-440 -> 40s -> flagged
    ]
    assert impl.solve(events) == ["iris"]


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")
